# MNIST — Thrust + cuBLAS （CPU）

**使用ライブラリ / libraries:** `cuBLAS + Thrust (+ CUDA runtime)`  
**バックエンド / backend:** `CPU (cudnn_cpp shims / g++)`

## これは何？ / What is this?

GPUを持っていなくてもGPUプログラミングを学べる教材です。ポイントは **同じ C++ ソースが、
コンパイル方法を変えるだけで CPU でも GPU でも動く** こと。CPUでは [cudnn_cpp](https://github.com/yomei-o/cudnn_cpp) のヘッダオンリー・シム（cuDNN/cuBLAS/Thrust/CUDA runtime のCPU実装）を、GPUでは本物のCUDAライブラリを使います。ソースコードは1行も変えません。

A hands-on way to learn GPU programming **without owning a GPU**. The key idea: the *same*
C++ source runs on **CPU and GPU** — only the compile command changes. On CPU it uses the
header-only shims from [cudnn_cpp](https://github.com/yomei-o/cudnn_cpp) (CPU implementations of cuDNN/cuBLAS/Thrust/the
CUDA runtime); on GPU it links the real CUDA libraries. Not one line of the source changes.

## この notebook の流れ / Steps
1. データ取得 → 2. ビルド → 3. 学習（`images/sec` を計測）→ 4. モデル保存 → 5. PNGを推論

## 注目ポイント / What to watch
- 学習ログの **`>> trained ... images/sec`**：CPUの1秒あたり学習枚数。GPU版と比べる基準になります。
- **精度 `TEST ACCURACY`** はCPUでもGPUでも（同じ計算なので）ほぼ同じになります。
  違うのは **速度** だけ — それがGPUを使う理由です。
  Accuracy is ~identical on CPU and GPU (same math); only **speed** differs — that is *why* GPUs matter.


In [ ]:
# --- リポジトリを取得 / clone the repo ---
![ -d cudnn_cpp ] || git clone --depth 1 https://github.com/yomei-o/cudnn_cpp.git
%cd cudnn_cpp

In [ ]:
# --- MNIST データを取得 / download MNIST ---
import os, urllib.request, gzip, shutil
os.makedirs('examples/mnist/data', exist_ok=True)
base='https://ossci-datasets.s3.amazonaws.com/mnist/'
for f in ['train-images-idx3-ubyte','train-labels-idx1-ubyte','t10k-images-idx3-ubyte','t10k-labels-idx1-ubyte']:
    dst='examples/mnist/data/'+f
    if not os.path.exists(dst):
        urllib.request.urlretrieve(base+f+'.gz', dst+'.gz')
        with gzip.open(dst+'.gz','rb') as g, open(dst,'wb') as o: shutil.copyfileobj(g,o)
        os.remove(dst+'.gz')
print('MNIST ready:', os.listdir('examples/mnist/data'))

## ビルド / Build

CPUとGPUの違いは **この1コマンドだけ**。/ The CPU vs GPU difference is *this single command*.
```
g++ -std=c++17 -O3 -I. examples/mnist/mnist_mlp.cpp -o mnist_mlp
```


In [ ]:
# --- build ---
!g++ -std=c++17 -O3 -I. examples/mnist/mnist_mlp.cpp -o mnist_mlp
print('built: mnist_mlp')

## 学習 / Train
MLP(784→128→10) を全60,000枚で5エポック。CPU（素のシム）でも1〜2分程度で終わります。（`-DCUDNN_CPU_USE_EIGEN -DCUBLAS_CPU_USE_EIGEN` を付けると数倍速くなります／後述）


In [ ]:
# --- train ---
!./mnist_mlp --epochs 5 --batch 64 --lr 0.1 --save mnist_mlp.bin

In [ ]:
# --- 手書き風のPNGサンプルを生成 (0-9) / make black-on-white digit PNGs ---
!g++ -std=c++17 -O2 -I. examples/mnist/make_samples.cpp -o mk && mkdir -p examples/mnist/samples && ./mk
import matplotlib.pyplot as plt, matplotlib.image as mpimg
fig,ax=plt.subplots(1,10,figsize=(15,2))
for d in range(10): ax[d].imshow(mpimg.imread(f'examples/mnist/samples/digit{d}.png'),cmap='gray'); ax[d].set_title(str(d)); ax[d].axis('off')
plt.show()

In [ ]:
# --- 保存したモデルでPNGを推論 / infer a PNG with the trained model ---
!./mnist_mlp --load mnist_mlp.bin --infer examples/mnist/samples/digit7.png
import matplotlib.pyplot as plt, matplotlib.image as mpimg
plt.imshow(mpimg.imread('examples/mnist/samples/digit7.png'),cmap='gray'); plt.axis('off'); plt.title('input PNG'); plt.show()

In [ ]:
# --- 自分で描いた数字をアップロードして推論 (任意) / upload your own digit (optional) ---
from google.colab import files
up = files.upload()   # 28x28 以上の白背景に黒で数字を描いたPNG推奨
for name in up:
    !./mnist_mlp --load mnist_mlp.bin --infer "$name"
    import matplotlib.pyplot as plt, matplotlib.image as mpimg
    plt.imshow(mpimg.imread(name),cmap='gray'); plt.axis('off'); plt.show()

## まとめ / Takeaway

このノートと **もう一方（mnist_thrust_cublas_gpu.ipynb）** を実行して、`images/sec` を比べてみてください。
同じソース・同じ精度で、速度だけが変わります。モデルやバッチを大きくすると差はさらに開きます。

Run this notebook and its **counterpart (mnist_thrust_cublas_gpu.ipynb)** and compare `images/sec`:
same source, same accuracy, different speed. The gap grows as the model/batch grows.

### やってみよう / Try it
- `--batch 256` や隠れ層/チャンネルを増やして、CPUとGPimages/secがどう変わるか観察。
- Increase `--batch`, widen the model — watch how CPU throughput drops while GPU stays flat.
